In [ ]:
import google.generativeai as genai

genai.configure(api_key="")

for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

# Reflection and Blogpost Writing

## Setup

In [30]:
from utils import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()
#print(OPENAI_API_KEY)

llm_config = {"model": "gemini-2.5-flash","api_key":"AQ.Ab8RN6Lzm_vYWb65hKLaCLzdmUse2uOPXF6TQ1HM8PS8eENwVw","api_type":"google"}


The task!


In [31]:
task = '''
        Write a concise but engaging blogpost about
      Mugil soft solutions . Make sure the blogpost is
       within 100 words.
       '''

Create a writer agent

In [32]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [33]:
reply = writer.generate_reply(messages=[{"content": task,"role": "user"}])


In [34]:
print(reply)

{'content': '## Unlock Your Digital Potential with Mugil Soft Solutions\n\nStruggling to navigate the complex digital landscape? Mugil Soft Solutions is your trusted partner. We craft innovative software and provide expert IT services designed to streamline your operations and boost your growth. From custom applications to robust cloud solutions, our dedicated team delivers precision and reliability. Experience the difference of seamless technology – transform your business today with Mugil Soft Solutions!', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [35]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [36]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Mugil soft solutions . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

## Mugil Soft Solutions: Your Digital Edge

Facing tech hurdles? Mugil Soft Solutions delivers innovative answers. We craft bespoke software – from intuitive web apps to robust enterprise systems – tailored to your unique business challenges. Our focus: seamless integration, scalable performance, and user-centric design. We build more than software; we create powerful solutions that empower growth, streamline operations, and transform your vision into reality. Partner with us. Discover the tailored technology that drives success.

--------------------------------------------------------------------------------
Critic (to Writer):

Okay, let's put on the critic's hat and dive into "Mugil Soft Solutions: Your Digital Edge."

**

In [37]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [38]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [39]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [40]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [41]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [42]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [43]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Mugil soft solutions . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

## Mugil Soft Solutions: Your Digital Edge

Facing tech hurdles? Mugil Soft Solutions delivers innovative answers. We craft bespoke software – from intuitive web apps to robust enterprise systems – tailored to your unique business challenges. Our focus: seamless integration, scalable performance, and user-centric design. We build more than software; we create powerful solutions that empower growth, streamline operations, and transform your vision into reality. Partner with us. Discover the tailored technology that drives success.

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

**********

## Get the summary

In [44]:
print(res.summary)

## Unlock Your Potential with Mugil Soft Solutions

Is generic software hindering your growth? Mugil Soft Solutions crafts custom digital solutions designed precisely for your business. We build powerful web applications, intuitive mobile experiences, and streamlined enterprise systems that integrate seamlessly. Our focus isn't just code; it's empowering your operations, boosting efficiency, and transforming challenges into opportunities. Partner with us to unleash innovation and drive measurable results. Experience technology that truly works for you.
